# Notebook 4: Análisis de Errores e Informe Técnico

Contenido:
1. Matrices de confusión por modelo
2. Análisis de errores más frecuentes
3. Visualización de ejemplos mal clasificados
4. Recomendaciones para mejora
5. Informe técnico final

In [1]:
import json, os, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report
import cv2

# Detectar entorno Colab vs Local
try:
    from google.colab import drive; IN_COLAB = True
    sys.path.insert(0, '/content/drive/MyDrive/TRADUCTOR_LSP')
except ImportError:
    IN_COLAB = False
    from pathlib import Path
    _nb_path = Path.cwd()
    _base = str(_nb_path.parent if _nb_path.name == 'notebooks' else _nb_path)
    sys.path.insert(0, _base)
    import os; os.chdir(_base)
from src.models import CNNLSTM, STGCN
from src.dataset.lsp_dataset import get_dataloaders
from src.training.metrics import plot_confusion_matrix, compute_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
with open('data/label2idx.json') as f:
    label2idx = json.load(f)
idx2label = {int(v): k for k, v in label2idx.items()}
N_CLASSES = len(label2idx)
CLASS_NAMES = [idx2label[i] for i in range(N_CLASSES)]

print(f'Clases: {CLASS_NAMES}')

Clases: ['vineta_002', 'vineta_003', 'vineta_004', 'vineta_005', 'vineta_006', 'vineta_008', 'vineta_011', 'vineta_012', 'vineta_013', 'vineta_014', 'vineta_015', 'vineta_017', 'vineta_018', 'vineta_019', 'vineta_020', 'vineta_021', 'vineta_022', 'vineta_023', 'vineta_024', 'vineta_025', 'vineta_027', 'vineta_029', 'vineta_030', 'vineta_037', 'vineta_042', 'vineta_043']


In [2]:
# ── Función de inferencia con predicciones detalladas ─────────────────
def get_predictions(model, loader, device, mode='pixels'):
    model.eval()
    all_preds, all_labels, all_probs, all_paths = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader):
            labels = batch['label']
            
            if mode == 'pixels':
                logits = model(batch['pixels'].to(device))
            else:
                logits = model(batch['landmarks'].to(device))
            
            import torch.nn.functional as F
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=1)
            
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs)
            if 'clase' in batch:
                all_paths.extend(batch['clase'])
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs), all_paths

In [3]:
# ── Cargar modelos ────────────────────────────────────────────────────
models_config = [
    ('CNN-LSTM', 'cnn_lstm', 'pixels'),
    ('ST-GCN',   'stgcn',    'landmarks'),
]

loaders_pixels = get_dataloaders(
    'data/manifest_segments.csv', label2idx, 'pixels',
    pixels_dir='data/processed_pixels', batch_size=16, num_workers=2,
)
loaders_kp = get_dataloaders(
    'data/manifest_segments.csv', label2idx, 'landmarks',
    landmarks_dir='data/landmarks', batch_size=32, num_workers=2,
)

all_results = {}

for model_name, ckpt_name, mode in models_config:
    print(f'\nAnalizando: {model_name}')
    
    if model_name == 'CNN-LSTM':
        model = CNNLSTM(n_classes=N_CLASSES, pretrained=False)
    else:
        model = STGCN(n_classes=N_CLASSES, n_nodes=75)
    
    ckpt_path = f'checkpoints/{ckpt_name}_best.pt'
    if not os.path.exists(ckpt_path):
        print(f'  Checkpoint no encontrado: {ckpt_path}')
        continue
    
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(DEVICE)
    
    loader = loaders_pixels['test'] if mode == 'pixels' else loaders_kp['test']
    y_true, y_pred, y_probs, clases = get_predictions(model, loader, DEVICE, mode)
    
    metrics = compute_metrics(y_true, y_pred, y_probs, N_CLASSES)
    all_results[model_name] = {
        'y_true': y_true, 'y_pred': y_pred, 'y_probs': y_probs,
        'metrics': metrics, 'clases': clases,
    }
    print(f'  Accuracy: {metrics["accuracy"]:.4f} | F1-macro: {metrics["f1_macro"]:.4f}')


Analizando: CNN-LSTM


RuntimeError: Error(s) in loading state_dict for CNNLSTM:
	Missing key(s) in state_dict: "cnn.0.weight", "cnn.1.weight", "cnn.1.bias", "cnn.1.running_mean", "cnn.1.running_var", "cnn.4.0.conv1.weight", "cnn.4.0.bn1.weight", "cnn.4.0.bn1.bias", "cnn.4.0.bn1.running_mean", "cnn.4.0.bn1.running_var", "cnn.4.0.conv2.weight", "cnn.4.0.bn2.weight", "cnn.4.0.bn2.bias", "cnn.4.0.bn2.running_mean", "cnn.4.0.bn2.running_var", "cnn.4.0.conv3.weight", "cnn.4.0.bn3.weight", "cnn.4.0.bn3.bias", "cnn.4.0.bn3.running_mean", "cnn.4.0.bn3.running_var", "cnn.4.0.downsample.0.weight", "cnn.4.0.downsample.1.weight", "cnn.4.0.downsample.1.bias", "cnn.4.0.downsample.1.running_mean", "cnn.4.0.downsample.1.running_var", "cnn.4.1.conv1.weight", "cnn.4.1.bn1.weight", "cnn.4.1.bn1.bias", "cnn.4.1.bn1.running_mean", "cnn.4.1.bn1.running_var", "cnn.4.1.conv2.weight", "cnn.4.1.bn2.weight", "cnn.4.1.bn2.bias", "cnn.4.1.bn2.running_mean", "cnn.4.1.bn2.running_var", "cnn.4.1.conv3.weight", "cnn.4.1.bn3.weight", "cnn.4.

In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────
for model_name, res in all_results.items():
    fig = plot_confusion_matrix(
        res['y_true'], res['y_pred'],
        class_names=CLASS_NAMES,
        output_path=f'confusion_matrix_{model_name.lower().replace("-","_")}.png',
        normalize=True,
    )
    plt.suptitle(f'Matriz de Confusión — {model_name}', fontsize=14, y=1.01)
    plt.show()
    print(f'Guardada: confusion_matrix_{model_name}.png')

In [ ]:
# ── Análisis de errores más frecuentes ────────────────────────────────
for model_name, res in all_results.items():
    print(f'\n{"="*50}')
    print(f'ANÁLISIS DE ERRORES — {model_name}')
    print('='*50)
    
    y_true = res['y_true']
    y_pred = res['y_pred']
    
    # Pares verdadero→predicho más comunes
    errors = [(idx2label[t], idx2label[p]) for t, p in zip(y_true, y_pred) if t != p]
    from collections import Counter
    error_counts = Counter(errors).most_common(10)
    
    print('Top 10 confusiones más frecuentes:')
    print(f'{"Verdadero":20s} → {"Predicho":20s} | Count')
    print('-' * 55)
    for (true_cls, pred_cls), count in error_counts:
        print(f'{true_cls:20s} → {pred_cls:20s} | {count}')
    
    # Clases con menor precisión
    report = classification_report(
        y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )
    df_report = pd.DataFrame(report).T
    worst_classes = df_report.iloc[:N_CLASSES].nsmallest(5, 'f1-score')[['precision', 'recall', 'f1-score', 'support']]
    print(f'\nClases con menor F1-score:')
    print(worst_classes.round(3).to_string())

In [ ]:
# ── Visualización de ejemplos mal clasificados ────────────────────────
# Mostrar frames de videos donde el modelo se equivocó
df_test = pd.read_csv('data/manifest_segments.csv')
df_test = df_test[df_test['split'] == 'test'].reset_index(drop=True)

if all_results:
    model_name = list(all_results.keys())[0]
    res = all_results[model_name]
    y_true = res['y_true']
    y_pred = res['y_pred']
    y_probs = res['y_probs']
    
    # Seleccionar 6 errores con alta confianza (casos más difíciles)
    error_mask = (y_true != y_pred)
    error_confs = y_probs[error_mask].max(axis=1)
    error_indices = np.where(error_mask)[0]
    top_errors = error_indices[np.argsort(error_confs)[::-1][:6]]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for plot_i, err_i in enumerate(top_errors[:6]):
        if err_i >= len(df_test):
            continue
        video_path = df_test.iloc[err_i]['ruta']
        true_cls = idx2label[y_true[err_i]]
        pred_cls = idx2label[y_pred[err_i]]
        conf     = y_probs[err_i][y_pred[err_i]] * 100
        
        # Extraer frame central del video
        try:
            cap = cv2.VideoCapture(video_path)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
            ret, frame = cap.read()
            cap.release()
            
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                axes[plot_i].imshow(frame_rgb)
        except Exception:
            axes[plot_i].set_facecolor('#f0f0f0')
        
        axes[plot_i].set_title(
            f'Verdadero: {true_cls}\nPredicho: {pred_cls} ({conf:.0f}%)',
            fontsize=9, color='red' if true_cls != pred_cls else 'green'
        )
        axes[plot_i].axis('off')
    
    plt.suptitle(f'Errores más confiados — {model_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('error_analysis_frames.png', dpi=120)
    plt.show()

In [ ]:
# ── Informe técnico final ─────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════╗')
print('║          INFORME TÉCNICO FINAL — SISTEMA LSP                ║')
print('╠══════════════════════════════════════════════════════════════╣')

for model_name, res in all_results.items():
    m = res['metrics']
    print(f'║ {model_name:12s} | Acc: {m["accuracy"]:.4f} | F1-macro: {m["f1_macro"]:.4f}              ║')

print('╠══════════════════════════════════════════════════════════════╣')
print('║ RECOMENDACIONES PARA MEJORAR PRECISIÓN:                     ║')
print('║  1. Aumentar dataset: más muestras de clases con F1 < 0.7   ║')
print('║  2. Video augmentation: flip, velocidad, ruido              ║')
print('║  3. Ajuste fino por contexto: palabras funcionalmente siml. ║')
print('║  4. Ensemble: combinar CNN-LSTM + ST-GCN con voting         ║')
print('║  5. VideoMAE: fine-tuning con más épocas y LR más baja      ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║ DESPLIEGUE:                                                 ║')
print('║  - API FastAPI: python api/main.py                          ║')
print('║  - Demo Gradio: python demo/app_gradio.py                   ║')
print('║  - HuggingFace Spaces: subir demo/app_gradio.py como app.py ║')
print('║  - Latencia objetivo: <200ms con modelo ONNX en CPU         ║')
print('╚══════════════════════════════════════════════════════════════╝')